# Stage 00a.0 — EPO OPS Coverage Probe

**What this notebook does:**  
For each of the 1 639 rows in the PatSeer Excel, uses the **Simple Family ID** to query EPO OPS and records how many drawing pages any family member can serve.  
The strategy: call `published-data/family/inpadoc/{family_id}` to get all family members, pick the first EP or WO publication, then probe that publication's `/images` endpoint.

The result is `epo_coverage_report.csv`, which downstream notebooks use to decide whether to fetch images from EPO or fall back to Google Patents.

---

## Why family ID instead of patent ID

PatSeer `Record Number` values (e.g. `US2022267016A1`) are US-format publication numbers.  
EPO OPS's `epodoc` normalisation doesn't reliably resolve these — probing by `Record Number` alone yields ~0% coverage even for patents that EPO holds.  
The `Simple Family ID` is a numeric INPADOC family ID that EPO OPS understands natively, and the family endpoint exposes all EP/WO members that *do* have EPO image pages.

---

## Where it fits in the pipeline

```
00a.0  ← YOU ARE HERE  (probe EPO coverage via family ID; build fallback list)
 ↓
00a    (download images — EPO for covered families, Google Patents for the rest)
 ↓
00a.1  (audit: which patents are missing / need re-download)
 ↓
00a.2  (SigLIP triage filter)
 ↓
00b    (figure crop + label matching)
```

---

## Cell guide

| Cell | What it does |
|------|--------------|
| 1 | Load config, credentials, EPO OPS client |
| 2 | Load patent IDs + Simple Family IDs from PatSeer Excel |
| 3 | Probe loop — family lookup → EP/WO member → image count; save `epo_coverage_report.csv` |
| 4 | Summary — coverage table, fallback count, country-code breakdown |

---

## Before you run

- **EPO OPS credentials required** — register a free app at https://developers.epo.org/ and add `EPO_CLIENT_KEY` / `EPO_CLIENT_SECRET` to `.env`.
- Cell 3 makes up to 2× requests per unique family (family lookup + image probe) at 4 req/s ≈ **14 minutes** worst case.
- Re-running Cell 3 is safe — the CSV is overwritten but the probe is stateless.

In [ ]:
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir      = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from config_loader import load_config
from extractor import build_epo_client

cfg    = load_config()
client = build_epo_client(cfg)

print("Config loaded.")
print("  patseer_excel:", cfg["paths"]["patseer_excel"])
print("  data dir     :", cfg["paths"]["data"])
print("EPO OPS token acquired successfully.")

In [ ]:
import pandas as pd

excel_path = cfg["paths"]["patseer_excel"]
raw_df     = pd.read_excel(excel_path, dtype=str)

# Keep Record Number + Simple Family ID; drop rows missing either
work_df = (
    raw_df[["Record Number", "Simple Family ID"]]
    .rename(columns={"Record Number": "patent_id", "Simple Family ID": "family_id"})
    .dropna(subset=["patent_id"])
    .assign(
        patent_id=lambda d: d["patent_id"].str.strip(),
        family_id=lambda d: d["family_id"].str.strip(),
    )
    .reset_index(drop=True)
)

patent_ids = work_df["patent_id"].tolist()
family_ids = work_df["family_id"].tolist()

n_with_family = work_df["family_id"].notna().sum()
n_unique_fam  = work_df["family_id"].nunique()

print(f"Total rows          : {len(work_df)}")
print(f"Rows with family ID : {n_with_family}")
print(f"Unique family IDs   : {n_unique_fam}")
print(f"Example patent_id   : {patent_ids[:3]}")
print(f"Example family_id   : {family_ids[:3]}")

In [ ]:
import time
import xml.etree.ElementTree as ET
import requests

_OPS_NS  = "http://www.epo.org/exchange-ops"
_EXCH_NS = "http://www.epo.org/exchange"

# Preferred country codes for the image probe — EP and WO have the best EPO image coverage
_PREFERRED_CC = ("EP", "WO", "DE", "FR", "GB")


def _pick_member(root) -> str | None:
    """
    From the family XML response, pick the best publication number to probe.
    Preference order: EP > WO > DE > FR > GB > any other member.
    Returns a docdb-style string like 'EP4049930A1', or None if no members found.
    """
    members = []  # list of (country_code, doc_number, kind_code)
    for pub_ref in root.iter(f"{{{_EXCH_NS}}}publication-reference"):
        doc_id = pub_ref.find(f"{{{_EXCH_NS}}}document-id")
        if doc_id is None:
            continue
        cc   = (doc_id.findtext(f"{{{_EXCH_NS}}}country")   or "").strip()
        num  = (doc_id.findtext(f"{{{_EXCH_NS}}}doc-number") or "").strip()
        kind = (doc_id.findtext(f"{{{_EXCH_NS}}}kind")       or "").strip()
        if cc and num:
            members.append((cc, num, kind))

    if not members:
        return None

    for preferred_cc in _PREFERRED_CC:
        for cc, num, kind in members:
            if cc == preferred_cc:
                return f"{cc}{num}{kind}" if kind else f"{cc}{num}"

    cc, num, kind = members[0]
    return f"{cc}{num}{kind}" if kind else f"{cc}{num}"


def _image_page_count(pub_id: str, client) -> int:
    """Return drawing page count for a resolved publication ID, or 0 on failure."""
    try:
        resp = client.get(f"published-data/publication/epodoc/{pub_id}/images")
        root = ET.fromstring(resp.text)
        for inst in root.iter(f"{{{_OPS_NS}}}document-instance"):
            if inst.attrib.get("desc", "").lower() == "drawing":
                return int(inst.attrib.get("number-of-pages", 0))
        return 0
    except Exception:
        return 0


def probe_epo_coverage(work_df: pd.DataFrame, client, cfg) -> pd.DataFrame:
    """
    For each row, use Simple Family ID to find an EP/WO member via the
    EPO OPS family endpoint, then probe that member's image page count.

    Returns a DataFrame with columns:
        patent_id      – original Record Number
        family_id      – Simple Family ID used for the lookup
        probed_pub_id  – the EP/WO publication actually queried (or empty)
        epo_pages      – drawing pages found (0 = none; -1 = error)
        epo_status     – OK | NO_DRAWINGS | NOT_FOUND | NO_FAMILY_ID | ERROR
        needs_fallback – True when epo_status != 'OK'
    """
    rows   = []
    n_ok   = 0
    n_miss = 0
    n_err  = 0
    total  = len(work_df)

    # Cache family lookups — multiple PatSeer rows can share the same family ID
    family_cache: dict[str, str | None] = {}  # family_id -> probed_pub_id or None

    for i, (_, row) in enumerate(work_df.iterrows(), start=1):
        patent_id = row["patent_id"]
        family_id = row["family_id"]

        if pd.isna(family_id) or not str(family_id).strip():
            rows.append({
                "patent_id"     : patent_id,
                "family_id"     : "",
                "probed_pub_id" : "",
                "epo_pages"     : -1,
                "epo_status"    : "NO_FAMILY_ID",
                "needs_fallback": True,
            })
            n_err += 1
            if i % 50 == 0 or i == total:
                print(f"[{i}/{total}]  OK: {n_ok}  MISSING: {n_miss}  ERROR: {n_err}")
            continue

        family_id = str(family_id).strip()
        pages      = -1
        status     = "ERROR"
        pub_id     = ""

        # Step 1 — resolve family members (cached)
        if family_id not in family_cache:
            try:
                resp = client.get(f"published-data/family/inpadoc/{family_id}")
                root = ET.fromstring(resp.text)
                family_cache[family_id] = _pick_member(root)
            except requests.exceptions.HTTPError as exc:
                family_cache[family_id] = None
                if exc.response is not None and exc.response.status_code == 404:
                    status = "NOT_FOUND"
                else:
                    status = "ERROR"
            except Exception:
                family_cache[family_id] = None
                status = "ERROR"
            time.sleep(0.25)

        pub_id = family_cache.get(family_id) or ""

        # Step 2 — probe the resolved EP/WO member for images
        if pub_id:
            pages  = _image_page_count(pub_id, client)
            status = "OK" if pages > 0 else "NO_DRAWINGS"
            time.sleep(0.25)
        elif status == "ERROR":  # family_cache resolved to None due to exception
            pages  = -1
            status = "NOT_FOUND" if status == "NOT_FOUND" else "ERROR"

        if status == "OK":
            n_ok  += 1
        elif status == "NOT_FOUND":
            n_miss += 1
        else:
            n_err += 1

        rows.append({
            "patent_id"     : patent_id,
            "family_id"     : family_id,
            "probed_pub_id" : pub_id,
            "epo_pages"     : pages,
            "epo_status"    : status,
            "needs_fallback": status != "OK",
        })

        if i % 50 == 0 or i == total:
            print(f"[{i}/{total}]  OK: {n_ok}  MISSING: {n_miss}  ERROR: {n_err}")

    return pd.DataFrame(rows)


coverage_df = probe_epo_coverage(work_df, client, cfg)

out_dir  = Path(cfg["paths"]["data"])
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "epo_coverage_report.csv"
coverage_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

In [ ]:
import re

total = len(coverage_df)

# ── Coverage table ────────────────────────────────────────────────────────────
status_counts = (
    coverage_df.groupby("epo_status", sort=False)
    .size()
    .rename("count")
    .reset_index()
)
status_counts["pct"] = (status_counts["count"] / total * 100).round(1)
status_counts = status_counts.sort_values("count", ascending=False).reset_index(drop=True)

print("=" * 42)
print(f"  EPO OPS Coverage  ({total} patents)")
print("=" * 42)
for _, row in status_counts.iterrows():
    print(f"  {row['epo_status']:<14} {row['count']:>5}  ({row['pct']:.1f}%)")
print("=" * 42)

# ── Fallback count ────────────────────────────────────────────────────────────
n_fallback = coverage_df["needs_fallback"].sum()
print(f"\nPatents needing Google Patents fallback: {n_fallback} / {total}  "
      f"({n_fallback / total * 100:.1f}%)")

# ── Country-code breakdown for fallback patents ───────────────────────────────
fallback_df = coverage_df[coverage_df["needs_fallback"]].copy()

def _country_code(pid: str) -> str:
    m = re.match(r"^([A-Z]{2})", str(pid))
    return m.group(1) if m else "??"

fallback_df["cc"] = fallback_df["patent_id"].map(_country_code)
top_cc = (
    fallback_df["cc"]
    .value_counts()
    .head(10)
    .rename_axis("country_code")
    .reset_index(name="count")
)
top_cc["pct_of_fallback"] = (top_cc["count"] / max(1, n_fallback) * 100).round(1)

print("\nTop 10 country-code prefixes among needs_fallback patents:")
print(top_cc.to_string(index=False))